In [2]:
pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import duckdb

In [4]:
#loading tables
duckdb.sql("CREATE TABLE discharges AS SELECT * FROM 'Inmate_Discharges_20260716.csv'")
duckdb.sql("CREATE TABLE inmates AS SELECT * FROM 'Daily_Inmates_In_Custody_20260716.csv'")
duckdb.sql("CREATE TABLE admissions as SELECT * FROM 'Inmate_Admissions_20260716.csv'")

times admitted and total days served calculated using admissions and discharges data from 01-01-2014 for all inmates

In [7]:
#checking earliest entry for admissions table
duckdb.sql("""
    SELECT MIN(ADMITTED_DT)
    FROM admissions
    """)

┌─────────────────────┐
│  min(ADMITTED_DT)   │
│      timestamp      │
├─────────────────────┤
│ 2014-01-01 07:30:00 │
└─────────────────────┘

In [106]:
#checking earliest entry for discharges table
duckdb.sql("""
    SELECT MIN(DISCHARGED_DT)
    FROM discharges
    """)

┌─────────────────────┐
│ min(DISCHARGED_DT)  │
│      timestamp      │
├─────────────────────┤
│ 2014-01-01 00:13:00 │
└─────────────────────┘

In [86]:
# percentage suggest A = Asian, B = Black, W = White based on reports from NYC DOC 
# "DOC At a Glance" reports with race breakdowns
# all other entries will be grouped as O = Other 
# note: NYC DOC reports Hispanic ~34% but H is only 0.27% so it is unlikely to represent Hispanic
duckdb.sql("""
    SELECT RACE, 100*num_inmates/6578
    FROM(SELECT RACE, COUNT(INMATEID) as num_inmates
    FROM inmates
    GROUP BY 1)
    
""")

┌─────────┬──────────────────────────────┐
│  RACE   │ ((100 * num_inmates) / 6578) │
│ varchar │            double            │
├─────────┼──────────────────────────────┤
│ NULL    │           0.1976284584980237 │
│ A       │            2.949224688355123 │
│ B       │            58.26999087868653 │
│ H       │           0.2736394040741867 │
│ I       │          0.36485253876558227 │
│ M       │          0.12161751292186075 │
│ O       │              30.130738826391 │
│ U       │           0.6080875646093037 │
│ W       │            7.084220127698389 │
└─────────┴──────────────────────────────┘

In [15]:
#current inmates table before top charge grouping
duckdb.sql("""
    SELECT i.INMATEID, i.ADMITTED_DT,
    CASE 
        WHEN i.RACE = 'B' THEN 'BLACK'
        WHEN i.RACE = 'A' THEN 'ASIAN'
        WHEN i.RACE = 'W' THEN 'WHITE'
        ELSE 'OTHER/UNSPECIFIED' END as RACE,
    i.AGE, i.CUSTODY_LEVEL, i.BRADH, i.INMATE_STATUS_CODE, i.SRG_FLG, i.INFRACTION, i.TOP_CHARGE,
    times_admitted, (EXTRACT(DAY FROM(TIMESTAMP '2026-07-16 12:00:00'-i.ADMITTED_DT))+ds.days_served) as total_days_served,
    EXTRACT(DAY FROM(TIMESTAMP '2026-07-16 12:00:00'-i.ADMITTED_DT)) as ca_days_served,
    FROM inmates i
    
    LEFT JOIN (SELECT INMATEID, COUNT(ADMITTED_DT) as times_admitted
    FROM admissions
    GROUP BY 1) ta on i.INMATEID = ta.INMATEID
    
    LEFT JOIN (SELECT INMATEID, SUM(EXTRACT(DAY FROM (DISCHARGED_DT-ADMITTED_DT))) as days_served
    FROM discharges
    GROUP BY 1) ds on i.INMATEID = ds.INMATEID
""")

┌──────────┬─────────────────────┬───────────────────┬───────┬───────────────┬─────────┬────────────────────┬─────────┬────────────┬────────────┬────────────────┬───────────────────┬────────────────┐
│ INMATEID │     ADMITTED_DT     │       RACE        │  AGE  │ CUSTODY_LEVEL │  BRADH  │ INMATE_STATUS_CODE │ SRG_FLG │ INFRACTION │ TOP_CHARGE │ times_admitted │ total_days_served │ ca_days_served │
│  int64   │      timestamp      │      varchar      │ int64 │    varchar    │ varchar │      varchar       │ varchar │  varchar   │  varchar   │     int64      │      int128       │     int64      │
├──────────┼─────────────────────┼───────────────────┼───────┼───────────────┼─────────┼────────────────────┼─────────┼────────────┼────────────┼────────────────┼───────────────────┼────────────────┤
│    11942 │ 2026-05-18 20:32:05 │ WHITE             │    58 │ MIN           │ Y       │ DE                 │ N       │ N          │ 140.20     │             15 │              1284 │             58 │


In [9]:
#exporting all unique top_charges to analyze for grouping
duckdb.sql("""
  COPY(
    SELECT TOP_CHARGE
    FROM admissions
    GROUP BY 1
    ) TO 'top_charges.csv' (HEADER, DELIMITER ',')
""")

In [12]:
#categorizing top_charges
import re

def categorize_charge(charge):
    if charge is None or charge == '0':
        return 'Unknown/Missing'
    
    charge = charge.strip()
    
    # Flag placeholder codes
    if re.search(r'(777\.77|888\.88|999\.99|666\.66|000\.00)', charge):
        return 'Unspecified/Placeholder'
    
    #codes with '110-xxx' are for attempted xxx
    is_attempt = charge.startswith('110-')
    if is_attempt:
        charge = charge[4:]  # strip the "110-" prefix
    
    # Non-Penal-Law prefixes
    if charge.startswith('VTL'):
        return 'Traffic/DWI (VTL)' + (' - Attempted' if is_attempt else '')
    if charge.startswith('PHL'):
        return 'Public Health Law'
    if charge.startswith('AC') or re.match(r'^\d+-\d+', charge):  # e.g. 10-131
        return 'NYC Administrative Code'
    if charge.startswith('GBL'):
        return 'General Business Law'
    if charge.startswith('FOA'):
        return 'FOA (needs verification)'
    if charge in ('VOP', 'VOPB', 'VOCD', 'CCW', 'CIVIL', 'CO', 'ROW', 'INS', 'FED', 'C-COM', 'OJCW'):
        return 'Administrative/Non-statute code (needs verification)'
    
    # Extract article number for Penal Law
    match = re.match(r'^(\d+)', charge)
    if not match:
        return 'Unclassified'
    
    article = int(match.group(1))
    
    #defining code ranges for categorization
    ranges = {
        (100, 119): 'Inchoate Offenses',
        (120, 121): 'Assault/Strangulation',
        (125, 125): 'Homicide',
        (130, 130): 'Sex Offenses',
        (135, 135): 'Kidnapping/Coercion',
        (140, 140): 'Burglary/Trespass',
        (145, 145): 'Criminal Mischief',
        (150, 150): 'Arson',
        (155, 168): 'Theft/Larceny/Robbery',
        (170, 179): 'Forgery/Fraud',
        (180, 219): 'Bribery/Official Misconduct/Perjury',
        (220, 220): 'Controlled Substances',
        (221, 221): 'Marijuana',
        (225, 225): 'Gambling',
        (230, 230): 'Prostitution',
        (235, 235): 'Obscenity',
        (240, 249): 'Public Order',
        (250, 259): 'Privacy/Family Offenses',
        (260, 264): 'Offenses Involving Children',
        (265, 265): 'Weapons/Firearms',
        (270, 279): 'Public Safety/Other',
        (351, 385): 'Animal Cruelty/Animal Violation',
        (460, 460): 'Enterprise Corruption',
        (470, 470): 'Money Laundering',
        (490, 490.35): 'Terrorism',
        (490.37, 491): 'Bio/Chem Weapon',
        (496, 496): 'Government Corruption',
        (511, 511): 'Traffic/DWI (VTL)',
        (600, 600): 'Traffic/DWI (VTL)',
        (1192, 1192): 'Traffic/DWI (VTL)'
    }
    
    for (low, high), category in ranges.items():
        if low <= article <= high:
            return category + (' - Attempted' if is_attempt else '')
    
    return 'Unclassified'

# All unique codes categorized
#df = duckdb.sql("SELECT DISTINCT TOP_CHARGE, COUNT(*) as count FROM inmates GROUP BY TOP_CHARGE").df()
#df['category'] = df['TOP_CHARGE'].apply(categorize_charge)
#df.to_csv('top_charges_categorized.csv', index=False)

In [18]:
#convert to df
df = duckdb.sql("""
    SELECT i.INMATEID, i.ADMITTED_DT,
    CASE 
        WHEN i.RACE = 'B' THEN 'BLACK'
        WHEN i.RACE = 'A' THEN 'ASIAN'
        WHEN i.RACE = 'W' THEN 'WHITE'
        ELSE 'OTHER/UNSPECIFIED' END as RACE,
    i.AGE, i.CUSTODY_LEVEL, i.BRADH, i.INMATE_STATUS_CODE, i.SRG_FLG, i.INFRACTION, i.TOP_CHARGE,
    times_admitted, (EXTRACT(DAY FROM(TIMESTAMP '2026-07-16 12:00:00'-i.ADMITTED_DT))+ds.days_served) as total_days_served,
    EXTRACT(DAY FROM(TIMESTAMP '2026-07-16 12:00:00'-i.ADMITTED_DT)) as ca_days_served,
    FROM inmates i
    
    LEFT JOIN (SELECT INMATEID, COUNT(ADMITTED_DT) as times_admitted
    FROM admissions
    GROUP BY 1) ta on i.INMATEID = ta.INMATEID
    
    LEFT JOIN (SELECT INMATEID, SUM(EXTRACT(DAY FROM (DISCHARGED_DT-ADMITTED_DT))) as days_served
    FROM discharges
    GROUP BY 1) ds on i.INMATEID = ds.INMATEID
""").df()
df['TOP_CHARGE_CATEGORY'] = df['TOP_CHARGE'].apply(categorize_charge)
df.to_csv('inmates_in_custody.csv', index=False)

In [16]:
df.head()

,INMATEID,ADMITTED_DT,RACE,AGE,CUSTODY_LEVEL,BRADH,INMATE_STATUS_CODE,SRG_FLG,INFRACTION,TOP_CHARGE,times_admitted,total_days_served,ca_days_served,TOP_CHARGE_CATEGORY
0,27488,2026-07-14 18:24:45,OTHER/UNSPECIFIED,59,None,N,DEP,N,N,VOP,11,1317.0,1,Administrative/Non-statute code (needs verific...
1,77324,2026-07-01 16:50:00,BLACK,39,MED,N,CS,Y,N,None,18,657.0,14,Unknown/Missing
2,95079,2026-05-07 20:33:00,BLACK,60,MIN,Y,DE,N,N,240.75,14,779.0,69,Public Order
3,20165153,2026-02-21 11:04:13,BLACK,29,MAX,Y,DE,N,Y,130.35,6,273.0,145,Sex Offenses
4,20187945,2025-11-05 01:02:18,OTHER/UNSPECIFIED,26,MAX,Y,DE,N,Y,120.10,5,401.0,253,Assault/Strangulation


1192 - VTL
351, 385 - Animal cruelty/Animal violation